# Cal4 Controlled Scan: Cluster Center Response

Student notebook, **usual warning, you'll need to change some paths!**

This notebook uses the Cal4 scan runs from the August BL4S logbook. The idea is simple:

1. Put the beam near the center of Cal4.
2. Move the DESY table by known distances.
3. Reconstruct the cluster center from calorimeter signals.
4. Check whether the reconstructed cluster center moves by the expected amount.

The goal is not only to get one number. The goal is to understand which clustering choices make the reconstructed position more reliable.

This notebook is wired to the August test-beam files in this workspace:

- scan runs from `cal4_cluster_scan_runs.csv`,
- August normal-trigger slopes from `august_normal_trigger_calibration_constants.csv`,
- channel pedestals from `QDCConfig/QDC0_ch*_calibration.json`.

In [ ]:
from pathlib import Path
import csv
import json
import math
import statistics
import subprocess
import sys

try:
    import numpy as np
except Exception as exc:
    np = None
    print("NumPy import failed. Plotting/tuning cells need NumPy in the notebook kernel:", exc)

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
except Exception as exc:
    plt = None
    patches = None
    print("Matplotlib import failed. Visual cells need Matplotlib in the notebook kernel:", exc)

DATA_DIR = Path(".")
CONFIG_DIR = Path("QDCConfig")
SCAN_CSV = Path("cal4_cluster_scan_runs.csv")
AUGUST_NORMAL_CONSTANTS_CSV = Path("august_normal_trigger_calibration_constants.csv")
AUGUST_FS_CONSTANTS_CSV = Path("august_fs_tdc_calibration_constants.csv")

print("Notebook Python:", sys.executable)
print("Python version:", sys.version.split()[0])
root_check = subprocess.run(
    [sys.executable, "-c", "import ROOT; print(ROOT.gROOT.GetVersion())"],
    text=True,
    capture_output=True,
)
if root_check.returncode == 0:
    print("PyROOT check: OK, ROOT", root_check.stdout.strip())
else:
    print("PyROOT check: FAILED for this kernel.")
    print("Use a ROOT-compatible Python kernel before running the ROOT analysis cells.")
    if root_check.stderr.strip():
        print(root_check.stderr.strip().splitlines()[-1])

def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def print_rows(rows, columns=None, max_rows=30):
    rows = list(rows)[:max_rows]
    if not rows:
        print("(no rows)")
        return
    if columns is None:
        columns = list(rows[0])
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    print("  ".join("-" * widths[c] for c in columns))
    for row in rows:
        print("  ".join(str(row.get(c, "")).ljust(widths[c]) for c in columns))

SCAN_ROWS = read_csv_rows(SCAN_CSV)

# Positions are DESY-table positions from the logbook. They are useful because
# the layout drawing includes a visual spacer column, while the physical cell
# centers are about 10 cm apart.
CAL_POSITIONS = {
    "Cal4":  (31.472, -3.696),
    "Cal17": (21.699, -3.696),
    "Cal18": (41.754, -3.384),
    "Cal7":  (51.774, -3.439),
    "Cal5":  (31.854, -12.986),
    "Cal1":  (42.114, -12.986),
    "Cal10": (21.688, -13.168),
    "Cal12": (52.219, -13.080),
    "Cal0":  (21.059, 6.423),
    "Cal2":  (31.196, 6.335),
    "Cal14": (41.417, 6.395),
    "Cal11": (51.396, 6.274),
    "Cal8":  (21.094, 16.233),
    "Cal19": (31.509, 15.943),
    "Cal9":  (41.542, 15.943),
    "Cal13": (51.396, 15.766),
}

LAYOUT = [
    ["Cal10", "Cal5", None, "Cal1", "Cal12"],
    ["Cal17", "Cal4", None, "Cal18", "Cal7"],
    ["Cal0", "Cal2", None, "Cal14", "Cal11"],
    ["Cal8", "Cal19", None, "Cal9", "Cal13"],
]

CAL_TO_QDC = {
    "Cal0": 0, "Cal1": 1, "Cal2": 2, "Cal4": 3,
    "Cal5": 4, "Cal7": 5, "Cal8": 6, "Cal9": 7,
    "Cal10": 8, "Cal11": 9, "Cal12": 10, "Cal13": 11,
    "Cal14": 12, "Cal17": 13, "Cal18": 14, "Cal19": 15,
}

def load_august_slopes(path=AUGUST_NORMAL_CONSTANTS_CSV):
    slopes = {}
    if not path.exists():
        return slopes
    for row in read_csv_rows(path):
        try:
            slopes[row["cal"]] = {
                "slope": float(row["slope_mev_per_adc"]),
                "offset": float(row.get("offset_mev", 0.0) or 0.0),
                "source": str(path),
            }
        except Exception:
            continue
    return slopes

def load_qdc_calibrations(config_dir=CONFIG_DIR, slope_csv=AUGUST_NORMAL_CONSTANTS_CSV):
    august_slopes = load_august_slopes(slope_csv)
    out = {}
    for cal, qdc in CAL_TO_QDC.items():
        path = config_dir / f"QDC0_ch{qdc}_calibration.json"
        if not path.exists():
            out[cal] = {"pedestal": 0.0, "slope": 1.0, "offset": 0.0, "source": "fallback"}
            continue
        with path.open(encoding="utf-8") as handle:
            data = json.load(handle)
        slope_info = august_slopes.get(cal)
        slope = float(data.get("calibration", {}).get("slope", 1.0))
        offset = float(data.get("calibration", {}).get("offset", 0.0))
        source = str(path)
        if slope_info is not None and slope_info["slope"] > 0:
            slope = slope_info["slope"]
            offset = slope_info["offset"]
            source = f"pedestal: {path}; August slope: {slope_info['source']}"
        out[cal] = {
            "pedestal": float(data.get("pedestal", {}).get("value", 0.0)),
            "slope": slope,
            "offset": offset,
            "source": source,
        }
    return out

QDC_CALIBRATIONS = load_qdc_calibrations()

print("Scan rows:", len(SCAN_ROWS))
print("Default usable scan rows:", sum(row["use_by_default"] == "yes" and row["root_file_ready"] == "yes" for row in SCAN_ROWS))
print("Available scan ROOT files:", sum((DATA_DIR / f"{row['run']}.root").exists() for row in SCAN_ROWS if row["run"]))
print("Using August normal-trigger slopes from:", AUGUST_NORMAL_CONSTANTS_CSV)
print("Using channel pedestals from:", CONFIG_DIR)
print("Cal4 constants:", QDC_CALIBRATIONS["Cal4"])

## 1. What Was Scanned?

The logbook describes controlled Cal4 movements at 3 GeV and 1 GeV. The center point is the normal Cal4-centered position. Then the table is moved step by step.

The picture below shows:

- the calorimeter positions from the logbook,
- the Cal4 center,
- horizontal scan points,
- vertical scan points.

Important: the table coordinate and the beam position in detector coordinates can have opposite signs. In the analysis we compare both the signed movement and the absolute movement.

In [ ]:
def draw_cal4_scan_plan(rows=SCAN_ROWS):
    if plt is None:
        print("Matplotlib is not available in this kernel; cannot draw the scan plan.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    ax = axes[0]
    for name, (h, v) in CAL_POSITIONS.items():
        color = "#f4f4f4"
        edge = "#555555"
        if name == "Cal4":
            color = "#ffe08a"
            edge = "#8a5a00"
        rect = patches.Rectangle((h - 4.5, v - 4.5), 9, 9, facecolor=color, edgecolor=edge, linewidth=1.5)
        ax.add_patch(rect)
        ax.text(h, v, name, ha="center", va="center", fontsize=9)

    default_rows = [r for r in rows if r["use_by_default"] == "yes"]
    for r in default_rows:
        h = float(r["table_h_cm"])
        v = float(r["table_v_cm"])
        if r["axis"] == "horizontal":
            ax.scatter(h, v, marker="o", s=60, color="#2c7fb8")
        elif r["axis"] == "vertical":
            ax.scatter(h, v, marker="s", s=55, color="#d95f0e")
        elif r["axis"] == "center":
            ax.scatter(h, v, marker="*", s=180, color="#2ca25f")

    ax.set_aspect("equal", adjustable="box")
    ax.invert_yaxis()
    ax.set_xlabel("DESY table H [cm]")
    ax.set_ylabel("DESY table V [cm]")
    ax.set_title("Cal4 scan points from the logbook")
    ax.grid(True, alpha=0.25)

    ax = axes[1]
    for energy, marker, color in [(3000, "o", "#2c7fb8"), (1000, "s", "#d95f0e")]:
        selected = [
            r for r in default_rows
            if int(r["energy_mev"]) == energy and r["axis"] in ("horizontal", "vertical")
        ]
        ax.scatter(
            [float(r["table_dx_cm"]) for r in selected],
            [float(r["table_dy_cm"]) for r in selected],
            marker=marker,
            color=color,
            label=f"{energy//1000} GeV",
            s=70,
        )
        for r in selected:
            ax.text(float(r["table_dx_cm"]) + 0.05, float(r["table_dy_cm"]) + 0.05, r["scan_id"], fontsize=7)

    ax.axhline(0, color="black", linewidth=1)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Table H shift from Cal4 center [cm]")
    ax.set_ylabel("Table V shift from Cal4 center [cm]")
    ax.set_title("Controlled scan offsets")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.tight_layout()

draw_cal4_scan_plan()
print_rows(SCAN_ROWS, ["scan_id", "energy_mev", "run", "axis", "table_dx_cm", "table_dy_cm", "root_file_ready", "use_by_default", "status"], max_rows=40)

### Questions

1. Which scan points are horizontal?
2. Which scan points are vertical?
3. Why do we need a center run at each energy?
4. Which runs are marked as diagnostic, faulty, or corrected?
5. Why should we not blindly average all runs if the logbook says something changed during the run?

## 2. Cluster-Center Method

For every event, the notebook will:

1. Read QDC values from the ROOT file.
2. Subtract pedestals from `QDCConfig`.
3. Convert to calibrated energy using the August normal-trigger calibration constants.
4. Mark cells above a threshold.
5. Group neighboring hit cells into clusters.
6. Pick the main cluster.
7. Compute the cluster center as an energy-weighted position.

You can change the algorithm settings later and see what changes.

In [ ]:
import ROOT
ROOT.gROOT.SetBatch(True)

def branch_exists(tree, name):
    return tree.GetBranch(name) is not None

def event_value(tree, branch):
    return float(getattr(tree, branch))

def calibrated_cell_value(tree, cal, cluster_input="energy"):
    qdc = CAL_TO_QDC[cal]
    branch = f"QDC0_ch{qdc}"
    if not branch_exists(tree, branch):
        return 0.0
    valid_branch = f"{branch}_valid"
    if branch_exists(tree, valid_branch) and int(getattr(tree, valid_branch)) == 0:
        return 0.0

    raw = event_value(tree, branch)
    c = QDC_CALIBRATIONS[cal]
    signal = raw - c["pedestal"]
    if cluster_input == "raw":
        return raw
    if cluster_input == "signal":
        return max(0.0, signal)
    return max(0.0, c["offset"] + c["slope"] * signal)

def event_matrix(tree, cluster_input="energy"):
    matrix = []
    for row in LAYOUT:
        out_row = []
        for cal in row:
            out_row.append(0.0 if cal is None else calibrated_cell_value(tree, cal, cluster_input=cluster_input))
        matrix.append(out_row)
    return matrix

def find_clusters(matrix, threshold, seed_threshold=None, connectivity=8):
    if seed_threshold is None:
        seed_threshold = threshold
    n_rows = len(matrix)
    n_cols = len(matrix[0])
    used = [[False] * n_cols for _ in range(n_rows)]
    clusters = []
    if connectivity == 4:
        neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    else:
        neighbors = [
            (-1, -1), (-1, 0), (-1, 1),
            (0, -1),           (0, 1),
            (1, -1),  (1, 0),  (1, 1),
        ]

    for r in range(n_rows):
        for c in range(n_cols):
            if used[r][c] or matrix[r][c] <= seed_threshold or LAYOUT[r][c] is None:
                continue
            stack = [(r, c)]
            used[r][c] = True
            cells = []
            while stack:
                rr, cc = stack.pop()
                cal = LAYOUT[rr][cc]
                value = matrix[rr][cc]
                if cal is not None:
                    cells.append((cal, rr, cc, value))
                for dr, dc in neighbors:
                    nr, nc = rr + dr, cc + dc
                    if nr < 0 or nc < 0 or nr >= n_rows or nc >= n_cols:
                        continue
                    if used[nr][nc] or LAYOUT[nr][nc] is None:
                        continue
                    if matrix[nr][nc] <= threshold:
                        continue
                    used[nr][nc] = True
                    stack.append((nr, nc))
            if cells:
                clusters.append(cells)
    return clusters

def cluster_center(cluster, algorithm="log", log_weight_offset=4.5):
    total = sum(value for _, _, _, value in cluster)
    if total <= 0:
        return None
    weighted_h = 0.0
    weighted_v = 0.0
    weight_sum = 0.0
    for cal, _, _, value in cluster:
        weight = value
        if algorithm in ("log", "log_weighted"):
            weight = max(0.0, log_weight_offset + math.log(value / total))
        h, v = CAL_POSITIONS[cal]
        weighted_h += weight * h
        weighted_v += weight * v
        weight_sum += weight
    if weight_sum <= 0:
        return cluster_center(cluster, algorithm="linear", log_weight_offset=log_weight_offset)
    h0, v0 = CAL_POSITIONS["Cal4"]
    return {
        "h_cm": weighted_h / weight_sum,
        "v_cm": weighted_v / weight_sum,
        "x_from_cal4_cm": weighted_h / weight_sum - h0,
        "y_from_cal4_cm": weighted_v / weight_sum - v0,
        "energy_sum": total,
        "size": len(cluster),
        "cells": [cal for cal, _, _, _ in cluster],
    }

def choose_cluster(clusters, mode="largest"):
    if not clusters:
        return None
    if mode == "contains_cal4":
        with_cal4 = [cluster for cluster in clusters if any(cell[0] == "Cal4" for cell in cluster)]
        if with_cal4:
            return max(with_cal4, key=lambda cluster: sum(cell[3] for cell in cluster))
    return max(clusters, key=lambda cluster: sum(cell[3] for cell in cluster))

def analyze_one_run(
    run,
    threshold=100.0,
    seed_threshold=None,
    connectivity=8,
    algorithm="log",
    log_weight_offset=4.5,
    cluster_input="energy",
    cluster_choice="largest",
    max_events=None,
):
    path = DATA_DIR / f"{run}.root"
    if not path.exists():
        raise FileNotFoundError(path)
    root_file = ROOT.TFile.Open(str(path))
    tree = root_file.Get("RAWdata")
    if not tree:
        raise RuntimeError(f"No RAWdata tree in {path}")

    n_entries = int(tree.GetEntries())
    if max_events is None:
        max_events = n_entries
    n_loop = min(max_events, n_entries)
    centers = []
    for index in range(n_loop):
        tree.GetEntry(index)
        matrix = event_matrix(tree, cluster_input=cluster_input)
        clusters = find_clusters(matrix, threshold=threshold, seed_threshold=seed_threshold, connectivity=connectivity)
        selected = choose_cluster(clusters, mode=cluster_choice)
        if selected is None:
            continue
        center = cluster_center(selected, algorithm=algorithm, log_weight_offset=log_weight_offset)
        if center is not None:
            centers.append(center)
    root_file.Close()
    if not centers:
        return {"run": run, "events_read": n_loop, "clusters": 0}
    xs = [c["x_from_cal4_cm"] for c in centers]
    ys = [c["y_from_cal4_cm"] for c in centers]
    es = [c["energy_sum"] for c in centers]
    sizes = [c["size"] for c in centers]
    return {
        "run": run,
        "events_read": n_loop,
        "clusters": len(centers),
        "cluster_fraction": len(centers) / n_loop if n_loop else 0.0,
        "x_mean_cm": float(statistics.fmean(xs)),
        "y_mean_cm": float(statistics.fmean(ys)),
        "x_median_cm": float(statistics.median(xs)),
        "y_median_cm": float(statistics.median(ys)),
        "x_std_cm": float(statistics.pstdev(xs)),
        "y_std_cm": float(statistics.pstdev(ys)),
        "energy_median": float(statistics.median(es)),
        "size_median": float(statistics.median(sizes)),
    }

def analyze_scan(
    rows=SCAN_ROWS,
    threshold=100.0,
    seed_threshold=None,
    connectivity=8,
    algorithm="log",
    log_weight_offset=4.5,
    cluster_input="energy",
    cluster_choice="largest",
    max_events=20000,
    use_default_only=True,
):
    results = []
    for row in rows:
        if use_default_only and row["use_by_default"] != "yes":
            continue
        if row["root_file_ready"] != "yes" or not row["run"]:
            continue
        result = analyze_one_run(
            int(row["run"]),
            threshold=threshold,
            seed_threshold=seed_threshold,
            connectivity=connectivity,
            algorithm=algorithm,
            log_weight_offset=log_weight_offset,
            cluster_input=cluster_input,
            cluster_choice=cluster_choice,
            max_events=max_events,
        )
        merged = dict(row)
        merged.update(result)
        for key in ("energy_mev", "table_h_cm", "table_v_cm", "table_dx_cm", "table_dy_cm"):
            merged[key] = float(merged[key])
        results.append(merged)
        print(f"done {row['scan_id']:22s} run {row['run']} clusters {result.get('clusters', 0)}")
    return results

def add_relative_shifts(results):
    centers = {}
    for row in results:
        if row["axis"] == "center":
            centers[int(row["energy_mev"])] = row
    out = []
    for row in results:
        item = dict(row)
        center = centers.get(int(row["energy_mev"]))
        if center and "x_median_cm" in row and "x_median_cm" in center:
            item["measured_dx_cm"] = row["x_median_cm"] - center["x_median_cm"]
            item["measured_dy_cm"] = row["y_median_cm"] - center["y_median_cm"]
            item["abs_table_shift_cm"] = math.hypot(row["table_dx_cm"], row["table_dy_cm"])
            item["abs_measured_shift_cm"] = math.hypot(item["measured_dx_cm"], item["measured_dy_cm"])
        out.append(item)
    return out

def plot_scan_comparison(results, axis="horizontal"):
    if plt is None:
        print("Matplotlib is not available in this kernel; cannot draw comparison plots.")
        return
    selected = [r for r in results if r["axis"] == axis and "measured_dx_cm" in r]
    if not selected:
        print("No results for", axis)
        return
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for energy, color in [(3000, "#2c7fb8"), (1000, "#d95f0e")]:
        rows = [r for r in selected if int(r["energy_mev"]) == energy]
        if not rows:
            continue
        expected = [r["table_dx_cm"] if axis == "horizontal" else r["table_dy_cm"] for r in rows]
        measured = [r["measured_dx_cm"] if axis == "horizontal" else r["measured_dy_cm"] for r in rows]
        axes[0].scatter(expected, measured, label=f"{energy//1000} GeV", color=color)
        for r, x, y in zip(rows, expected, measured):
            axes[0].text(x, y, r["scan_id"], fontsize=7)
        axes[1].scatter([abs(x) for x in expected], [abs(y) for y in measured], label=f"{energy//1000} GeV", color=color)

    for ax in axes:
        lim = 5.5
        ax.plot([-lim, lim], [-lim, lim], "--", color="gray", label="same sign")
        ax.plot([-lim, lim], [lim, -lim], ":", color="gray", label="opposite sign")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.grid(True, alpha=0.25)
        ax.set_aspect("equal", adjustable="box")
    axes[0].set_xlabel(f"table {axis} shift [cm]")
    axes[0].set_ylabel(f"measured cluster {axis} shift [cm]")
    axes[0].set_title("Signed movement")
    axes[1].set_xlabel("expected movement magnitude [cm]")
    axes[1].set_ylabel("measured movement magnitude [cm]")
    axes[1].set_title("Magnitude response")
    axes[0].legend()
    plt.tight_layout()

def summarize_response(results, axis="horizontal"):
    rows = [r for r in results if r["axis"] == axis and "measured_dx_cm" in r]
    summary = []
    for energy in sorted({int(r["energy_mev"]) for r in rows}):
        selected = [r for r in rows if int(r["energy_mev"]) == energy]
        expected = [r["table_dx_cm"] if axis == "horizontal" else r["table_dy_cm"] for r in selected]
        measured = [r["measured_dx_cm"] if axis == "horizontal" else r["measured_dy_cm"] for r in selected]
        if len(expected) < 2:
            continue
        denominator = sum(x * x for x in expected)
        slope_same = sum(x * y for x, y in zip(expected, measured)) / denominator
        slope_opposite = sum((-x) * y for x, y in zip(expected, measured)) / denominator
        residual_same = math.sqrt(statistics.fmean((y - x) ** 2 for x, y in zip(expected, measured)))
        residual_opposite = math.sqrt(statistics.fmean((y + x) ** 2 for x, y in zip(expected, measured)))
        summary.append({
            "axis": axis,
            "energy_mev": energy,
            "n_points": len(selected),
            "slope_same_sign": slope_same,
            "slope_opposite_sign": slope_opposite,
            "rms_residual_same_cm": residual_same,
            "rms_residual_opposite_cm": residual_opposite,
            "better_sign": "same" if residual_same <= residual_opposite else "opposite",
        })
    return summary

## 3. Run the Default Analysis

The default settings are deliberately ordinary:

```text
threshold = 100 MeV
connectivity = 8 neighbors
position = log-weighted center
cluster choice = largest cluster
```

If this cell is slow, reduce `max_events` first. Once the workflow is correct, increase it.

In [ ]:
results = analyze_scan(
    threshold=100.0,
    seed_threshold=None,
    connectivity=8,
    algorithm="log",
    log_weight_offset=4.5,
    cluster_input="energy",
    cluster_choice="largest",
    max_events=20000,
)
results = add_relative_shifts(results)

columns = [
    "scan_id", "energy_mev", "axis", "table_dx_cm", "table_dy_cm",
    "measured_dx_cm", "measured_dy_cm", "clusters", "cluster_fraction",
]
print_rows(results, columns, max_rows=40)

## 4. Compare Expected and Reconstructed Motion

The plot compares the known table movement with the reconstructed cluster-center movement.

Look at both panels:

- signed movement tells you whether the coordinate convention is the same or opposite,
- magnitude response tells you whether the cluster center moves by the right distance.

In [ ]:
plot_scan_comparison(results, axis="horizontal")
plot_scan_comparison(results, axis="vertical")

summary = summarize_response(results, axis="horizontal") + summarize_response(results, axis="vertical")
print_rows(summary, ["axis", "energy_mev", "n_points", "slope_same_sign", "slope_opposite_sign", "rms_residual_same_cm", "rms_residual_opposite_cm", "better_sign"], max_rows=20)

### Questions

1. Does the cluster center move in the expected direction?
2. Does it move by the expected amount?
3. Is the response closer to correct at 3 GeV or 1 GeV?
4. Which scan point looks worst?
5. What could cause a reconstructed shift to be smaller than the table shift?

## 5. Play With the Clustering Code

Now change one setting at a time.

Try:

- `threshold`: 50, 100, 150, 200 MeV
- `seed_threshold`: `None`, 150, 250 MeV
- `connectivity`: 4 or 8
- `algorithm`: `"linear"` or `"log"`
- `log_weight_offset`: 3.5, 4.5, 5.5
- `cluster_choice`: `"largest"` or `"contains_cal4"`

Do not change everything at once. Change one setting, rerun, and compare the summary.

In [ ]:
test_results = analyze_scan(
    threshold=150.0,
    seed_threshold=None,
    connectivity=8,
    algorithm="log",
    log_weight_offset=4.5,
    cluster_input="energy",
    cluster_choice="largest",
    max_events=20000,
)
test_results = add_relative_shifts(test_results)
test_summary = summarize_response(test_results, axis="horizontal") + summarize_response(test_results, axis="vertical")
print_rows(test_summary, ["axis", "energy_mev", "n_points", "slope_same_sign", "slope_opposite_sign", "rms_residual_same_cm", "rms_residual_opposite_cm", "better_sign"], max_rows=20)

### Final Questions

1. Which setting gave the smallest residual?
2. Did a better residual also keep a high cluster-finding fraction?
3. What setting would you recommend for the physics analysis?
4. What additional plot would convince you that the algorithm is really better?